# 02 SQL Analysis
Loads the clean data into SQLite and runs the first 3 business queries. The queries are saved in the SQL folder.

In [1]:
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
print(os.getcwd())

C:\Saas-Sales-Profitability


In [2]:
%%writefile .gitignore
Data/
.ipynb_checkpoints/

Overwriting .gitignore


In [3]:
import sqlite3, pandas as pd
df = pd.read_csv("Data/Clean/saas_clean.csv")
df.columns = [c.replace(" ", "_") for c in df.columns]
conn = sqlite3.connect("Data/saas.db")
df.to_sql("sales", conn, if_exists="replace", index=False)

def run(path):
    with open(path) as f:
        return pd.read_sql(f.read(), conn)

pd.read_sql("SELECT COUNT(*) AS total_rows FROM sales", conn)

,total_rows
0,9994


In [4]:
%%writefile SQL/01_segment_profit.sql
-- Business question: Which segments make money, and at what margin?
SELECT Segment,
       COUNT(DISTINCT Order_ID) AS orders,
       ROUND(SUM(Sales),0) AS revenue,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(100.0*SUM(Profit)/SUM(Sales),1) AS margin_pct
FROM sales
GROUP BY Segment
ORDER BY profit DESC;

Overwriting SQL/01_segment_profit.sql


In [5]:
run("SQL/01_segment_profit.sql")

,Segment,orders,revenue,profit,margin_pct
0,SMB,2586,1161401.0,134119.0,11.5
1,Strategic,1514,706146.0,91979.0,13.0
2,Enterprise,909,429653.0,60299.0,14.0


In [6]:
%%writefile SQL/02_product_profit.sql
-- Business question: Which products make or lose money overall?
SELECT Product,
       COUNT(*) AS order_lines,
       ROUND(SUM(Sales),0) AS revenue,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(100.0*SUM(Profit)/SUM(Sales),1) AS margin_pct,
       ROUND(AVG(Discount)*100,1) AS avg_discount_pct
FROM sales
GROUP BY Product
ORDER BY profit ASC;

Overwriting SQL/02_product_profit.sql


In [7]:
run("SQL/02_product_profit.sql")

,Product,order_lines,revenue,profit,margin_pct,avg_discount_pct
0,Marketing Suite,228,114880.0,-3473.0,-3.0,21.1
1,Storage,217,3024.0,950.0,31.4,8.2
2,Big Ol Database,115,189239.0,3385.0,1.8,30.6
3,ChatBot Plugin,796,27119.0,6528.0,24.1,7.5
4,SaaS Connector Pack - Gold,254,16476.0,6964.0,42.3,8.0
5,ContactMatcher,1842,410378.0,12496.0,3.0,35.3
6,SaaS Connector Pack,957,91705.0,13059.0,14.2,13.8
7,OneView,466,107532.0,18138.0,16.9,16.7
8,Marketing Suite - Gold,846,223844.0,21279.0,9.5,7.5
9,FinanceHub,981,340935.0,32136.0,9.4,13.3


In [8]:
%%writefile SQL/03_discount_tier.sql
-- Business question: At which discount tier do orders start losing money?
SELECT Discount_Tier,
       COUNT(*) AS order_lines,
       ROUND(SUM(Sales),0) AS revenue,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(100.0*SUM(Profit)/SUM(Sales),1) AS margin_pct,
       ROUND(100.0*SUM(Is_Loss)/COUNT(*),1) AS loss_line_pct
FROM sales
GROUP BY Discount_Tier
ORDER BY MIN(Discount);

Overwriting SQL/03_discount_tier.sql


In [9]:
run("SQL/03_discount_tier.sql")

,Discount_Tier,order_lines,revenue,profit,margin_pct,loss_line_pct
0,0%,4798,1087908.0,320988.0,29.5,0.0
1,1-10%,94,54369.0,9029.0,16.6,4.3
2,11-20%,3709,792153.0,91756.0,11.6,14.0
3,21-30%,227,103227.0,-10369.0,-10.0,91.6
4,31-40%,233,130911.0,-25448.0,-19.4,88.8
5,40%+,933,128632.0,-99559.0,-77.4,100.0


In [10]:
%%writefile SQL/04_discount_exact.sql
-- Business question: At exactly which discount value do order lines start losing money?
SELECT ROUND(Discount*100,0) AS discount_pct,
       COUNT(*) AS order_lines,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(100.0*SUM(Is_Loss)/COUNT(*),1) AS loss_line_pct
FROM sales
GROUP BY Discount
ORDER BY Discount;

Overwriting SQL/04_discount_exact.sql


In [11]:
run("SQL/04_discount_exact.sql")

,discount_pct,order_lines,profit,loss_line_pct
0,0.0,4798,320988.0,0.0
1,10.0,94,9029.0,4.3
2,15.0,52,1419.0,32.7
3,20.0,3657,90337.0,13.7
4,30.0,227,-10369.0,91.6
5,32.0,27,-2391.0,100.0
6,40.0,206,-23057.0,87.4
7,45.0,11,-2493.0,100.0
8,50.0,66,-20506.0,100.0
9,60.0,138,-5945.0,100.0


In [12]:
%%writefile SQL/05_product_rank_by_segment.sql
-- Business question: Within each segment, which products rank best and worst on profit?
WITH seg_prod AS (
    SELECT Segment, Product, SUM(Sales) AS revenue, SUM(Profit) AS profit
    FROM sales
    GROUP BY Segment, Product
)
SELECT Segment, Product,
       ROUND(profit,0) AS profit,
       ROUND(100.0*profit/revenue,1) AS margin_pct,
       RANK() OVER (PARTITION BY Segment ORDER BY profit DESC) AS profit_rank
FROM seg_prod
ORDER BY Segment, profit_rank;

Overwriting SQL/05_product_rank_by_segment.sql


In [13]:
run("SQL/05_product_rank_by_segment.sql")

,Segment,Product,profit,margin_pct,profit_rank
0,Enterprise,Alchemy,12544.0,38.2,1
1,Enterprise,Site Analytics,8912.0,12.9,2
2,Enterprise,Data Smasher,8493.0,26.5,3
3,Enterprise,Support,8287.0,41.9,4
4,Enterprise,FinanceHub,5720.0,9.8,5
5,Enterprise,Marketing Suite - Gold,5044.0,11.6,6
6,Enterprise,OneView,3726.0,20.6,7
7,Enterprise,ContactMatcher,2758.0,3.9,8
8,Enterprise,SaaS Connector Pack,1632.0,9.6,9
9,Enterprise,SaaS Connector Pack - Gold,1129.0,40.9,10


In [14]:
%%writefile SQL/06_monthly_trend.sql
-- Business question: How do revenue and profit move month by month?
WITH monthly AS (
    SELECT Order_Month, SUM(Sales) AS revenue, SUM(Profit) AS profit
    FROM sales
    GROUP BY Order_Month
)
SELECT Order_Month,
       ROUND(revenue,0) AS revenue,
       ROUND(profit,0) AS profit,
       ROUND(SUM(revenue) OVER (ORDER BY Order_Month),0) AS running_revenue,
       ROUND(SUM(profit) OVER (ORDER BY Order_Month),0) AS running_profit,
       ROUND(100.0*(revenue - LAG(revenue) OVER (ORDER BY Order_Month))
             / LAG(revenue) OVER (ORDER BY Order_Month),1) AS mom_revenue_pct
FROM monthly
ORDER BY Order_Month;

Overwriting SQL/06_monthly_trend.sql


In [15]:
run("SQL/06_monthly_trend.sql")

,Order_Month,revenue,profit,running_revenue,running_profit,mom_revenue_pct
0,2020-01,13946.0,2447.0,13946.0,2447.0,NaN
1,2020-02,4811.0,866.0,18757.0,3312.0,-65.5
2,2020-03,55691.0,499.0,74448.0,3811.0,1057.7
3,2020-04,28295.0,3489.0,102743.0,7300.0,-49.2
4,2020-05,23648.0,2739.0,126391.0,10039.0,-16.4
5,2020-06,34595.0,4977.0,160987.0,15015.0,46.3
6,2020-07,33946.0,-841.0,194933.0,14174.0,-1.9
7,2020-08,27909.0,5318.0,222842.0,19492.0,-17.8
8,2020-09,81777.0,8328.0,304620.0,27820.0,193.0
9,2020-10,31453.0,3448.0,336073.0,31268.0,-61.5


In [33]:
%%writefile SQL/07_top_loss_combos.sql
-- Business question: Which 10 product-segment combinations earn the least profit (only the first is loss-making)?
SELECT Product, Segment,
       COUNT(*) AS order_lines,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(AVG(Discount)*100,1) AS avg_discount_pct
FROM sales
GROUP BY Product, Segment
ORDER BY SUM(Profit) ASC
LIMIT 10;

Overwriting SQL/07_top_loss_combos.sql


In [34]:
run("SQL/07_top_loss_combos.sql")

,Product,Segment,order_lines,profit,avg_discount_pct
0,Marketing Suite,SMB,131,-4436.0,22.1
1,Storage,Enterprise,44,121.0,6.8
2,Storage,Strategic,59,252.0,12.2
3,Marketing Suite,Enterprise,33,325.0,15.5
4,Big Ol Database,Enterprise,23,541.0,27.4
5,Storage,SMB,114,577.0,6.7
6,Marketing Suite,Strategic,64,638.0,21.9
7,Big Ol Database,Strategic,35,703.0,34.6
8,ChatBot Plugin,Enterprise,150,1069.0,6.8
9,SaaS Connector Pack - Gold,Enterprise,41,1129.0,8.3


In [25]:
%%writefile SQL/08_yearly_retention.sql
-- Business question: What share of customers active in a year were still active the next year?
WITH active AS (
    SELECT DISTINCT Customer_ID, Order_Year FROM sales
)
SELECT a.Order_Year,
       COUNT(*) AS active_customers,
       COUNT(n.Customer_ID) AS active_next_year,
       ROUND(100.0*COUNT(n.Customer_ID)/COUNT(*),1) AS retention_pct
FROM active a
LEFT JOIN active n
       ON a.Customer_ID = n.Customer_ID AND n.Order_Year = a.Order_Year + 1
GROUP BY a.Order_Year
ORDER BY a.Order_Year;


Overwriting SQL/08_yearly_retention.sql


In [26]:
run("SQL/08_yearly_retention.sql")

,Order_Year,active_customers,active_next_year,retention_pct
0,2020,99,99,100.0
1,2021,99,99,100.0
2,2022,99,99,100.0
3,2023,99,0,0.0


In [27]:
%%writefile SQL/09_loss_customers.sql
-- Business question: Which customers lose us money overall, and what discount do they get?
SELECT Customer,
       COUNT(DISTINCT Order_ID) AS orders,
       ROUND(SUM(Sales),0) AS revenue,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(AVG(Discount)*100,1) AS avg_discount_pct
FROM sales
GROUP BY Customer_ID, Customer
HAVING SUM(Profit) < 0
ORDER BY profit ASC;

Overwriting SQL/09_loss_customers.sql


In [28]:
run("SQL/09_loss_customers.sql")

,Customer,orders,revenue,profit,avg_discount_pct
0,Allstate,61,39942.0,-2796.0,12.4
1,Bosch,60,25445.0,-1824.0,16.5
2,Nissan Motor,40,22651.0,-1585.0,17.9
3,Costco Wholesale,37,21797.0,-1355.0,13.7
4,Walgreens,33,14421.0,-741.0,15.0
5,Sprint Nextel,51,18852.0,-463.0,17.0
6,Morgan Stanley,60,28631.0,-362.0,17.5
7,HonHai Precision Industry,40,18457.0,-137.0,13.0
8,HSBC Holdings,41,22704.0,-104.0,18.6
9,Gazprom,23,5403.0,-14.0,18.3


In [29]:
%%writefile SQL/10_discount_cap_summary.sql
-- Business question: How much profit do lines above a 20% discount destroy?
SELECT CASE WHEN Discount > 0.20 THEN 'Above 20%' ELSE '20% or less' END AS discount_band,
       COUNT(*) AS order_lines,
       ROUND(SUM(Sales),0) AS revenue,
       ROUND(SUM(Profit),0) AS profit,
       ROUND(100.0*SUM(Profit)/SUM(Sales),1) AS margin_pct,
       ROUND(100.0*SUM(Is_Loss)/COUNT(*),1) AS loss_line_pct
FROM sales
GROUP BY discount_band;

Overwriting SQL/10_discount_cap_summary.sql


In [30]:
run("SQL/10_discount_cap_summary.sql")

,discount_band,order_lines,revenue,profit,margin_pct,loss_line_pct
0,20% or less,8601,1934431.0,421773.0,21.8,6.1
1,Above 20%,1393,362770.0,-135376.0,-37.3,96.8
